# Day 9: Serving ML Models with FastAPI & Docker

## Why FastAPI?
FastAPI gives you request validation, async support and free interactive docs (Swagger UI) with far less boilerplate than Flask.

In [ ]:
# pip install fastapi uvicorn scikit-learn joblib

## Defining a Prediction Endpoint
Pydantic models describe and validate the expected request body automatically - a bad request is rejected before your code even runs.

In [ ]:
# app.py
from fastapi import FastAPI
from pydantic import BaseModel
import joblib

app = FastAPI(title="Iris Classifier API")
model = joblib.load("iris_model.pkl")   # trained & saved earlier, e.g. Day 75 of the advanced track

class IrisInput(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

@app.post("/predict")
def predict(features: IrisInput):
    X = [[features.sepal_length, features.sepal_width,
          features.petal_length, features.petal_width]]
    prediction = model.predict(X)[0]
    return {"predicted_class": int(prediction)}

@app.get("/health")
def health():
    return {"status": "ok"}

## Running & Testing the API
uvicorn runs the app; the auto-generated docs at /docs let you try requests straight from the browser.

In [ ]:
# terminal:
# uvicorn app:app --reload --port 8000
# then open http://localhost:8000/docs in a browser

# or test from Python:
import requests
payload = {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}
# response = requests.post("http://localhost:8000/predict", json=payload)
# print(response.json())

## Containerizing with Docker
A Dockerfile packages your API, model file and dependencies into one image that runs identically on any machine or cloud provider.

In [ ]:
# Dockerfile
# FROM python:3.11-slim
# WORKDIR /app
# COPY requirements.txt .
# RUN pip install --no-cache-dir -r requirements.txt
# COPY . .
# EXPOSE 8000
# CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

# terminal commands:
# docker build -t iris-api .
# docker run -p 8000:8000 iris-api

## From Local API to the Cloud
The same Docker image can be pushed to a container registry and deployed on any cloud - the workflow is the same everywhere.

In [ ]:
# typical deployment targets for a Dockerized model API:
# - Render / Railway / Fly.io  (simplest, good for student projects)
# - AWS ECS / App Runner
# - Google Cloud Run
# - Azure Container Apps

# general pattern:
# docker build -t iris-api .
# docker tag iris-api <registry>/iris-api:latest
# docker push <registry>/iris-api:latest
# <cloud provider> deploy <registry>/iris-api:latest